<a href="https://colab.research.google.com/github/raisa1521/Hands-on-Machine-Learning/blob/main/5%EC%9E%A5_%EC%86%8C%ED%94%84%ED%8A%B8_%EB%B2%A1%ED%84%B0_%EB%A8%B8%EC%8B%A0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **5장 서포트 벡터 머신**

**서포트 벡터 머신(SVM)** : 선형이나 비선형 분류, 회귀, 특이치 탐지에 사용할 수 있는 다목적 머신러닝 모델

- 중소규모의 비선형 데이터셋, 특히 분류 작업에서 잘 작동함
- 매우 큰 데이터셋으로는 잘 확장되지 않음

## **5.1 선형 SVM 분류**

SVM 분류기는 두 클래스를 나누면서 가장 가까운 훈련 샘플로부터 가능한 한 멀리 떨어진 결정 경계를 찾는다.

- 클래스 사이에 가능한 한 폭이 넓은 도로를 찾는 것  =  **라지 마진 분류**

- 도로 바깥쪽에 훈련 샘플을 추가해도 결정 경계에는 영향을 미치지 않는다.

**서포트 벡터** : 도로의 경계에 위치하여 결정 경계를 결정하는 훈련 샘플



SVM은 특성의 스케일에 민감

특성의 스케일을 조정하면 결정 경계가 좋아질 수 있음(사이킷런의 StandardScaler 사용)

### **5.1.1 소프트 마진 분류**

**하드 마진 분류** : 모든 샘플이 도로 바깥쪽에 올바르게 분류되는 경우

하드 마진 분류의 문제점

- 데이터가 선형적으로 구분될 수 있어야 제대로 작동
- 이상치에 민감

**마진 오류** : 샘플이 도로 중간이나 심지어 반대쪽에 위치하는 경우

**소프트 마진 분류** : 도로의 폭을 가능한 한 넓게 유지하는 것과 마진 오류 사이에 적절한 균형을 잡는 방법

**규제 하이퍼파라미터 C**

- C 감소 -> 도로가 넓어지고 마진 오류가 많아짐
- C 증가 -> 도로가 좁아지고 마진 오류가 적어짐
- C를 줄이면 도로를 지지하는 샘플이 많아지므로 과대적합 위험 감소
- C를 너무 많이 줄이면 과소적합

-> SVM 모델이 과대적합이라면 C를 감소시켜 규제할 수 있음

In [ ]:
# 붓꽃 데이터셋에서 Iris-Virginica 품종을 감지하는 선형 SVM 모델
from sklearn.datasets import load_iris
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

iris = load_iris(as_frame=True)

X = iris.data[["petal length (cm)", "petal width (cm)"]].values
y = (iris.target == 2)  # Iris-Virginica

svm_clf = make_pipeline(
    StandardScaler(),
    LinearSVC(C=1, dual=True, random_state=42)
)

svm_clf.fit(X, y)

Pipeline(steps=[('standardscaler', StandardScaler()),
                ('linearsvc', LinearSVC(C=1, dual=True, random_state=42))])

In [ ]:
X_new = [[5.5, 1.7], [5.0, 1.5]]

svm_clf.predict(X_new)

array([ True, False])

-첫번째 꽃은 Iris-Virginica로 분류되지만 두번째 꽃은 그렇지 않음

In [ ]:
svm_clf.decision_function(X_new)

array([ 0.66163411, -0.22036063])

-SVM 모델은 각 샘플과 결정 경계 사이의 거리를 양수 또는 음수로 측정

-> LinearSVC에는 클래스 확률을 추정하는 predict_proba() 메서드가 없음

-> LinearSVC 대신 SVC 클래스를 사용하고 probability=True로 설정하면 훈련이 끝난 후 SVM 결정 함수 점수를 추정 확률에 매핑하는 추가 모델을 훈련

-> 이 과정에서는 5-폴드 교차 검증을 사용해서 훈련 속도가 느려짐

이후 다음 메서드를 사용 가능

- predict_proba()
- predict_log_proba()

## **5.2 비선형 SVM 분류**

선형 SVM 분류기는 효율적이지만 선형적으로 분류할 수 없는 데이터셋이 많음

비선형 데이터셋을 다루는 방법 => 다항 특성과 같은 특성을 추가하는 것

-> 특성을 추가하여 원래 선형적으로 구분할 수 없던 데이터셋을 선형적으로 구분할 수 있게 만들 수 있음

In [ ]:
#다항 특성을 사용한 선형 SVM
#moons 데이터셋 : 마주 보는 두 개의 초승달 모양으로 데이터 포인트가 놓인 이진 분류 데이터셋

from sklearn.datasets import make_moons
from sklearn.preprocessing import PolynomialFeatures

X, y = make_moons(
    n_samples=100,
    noise=0.15,
    random_state=42
)

polynomial_svm_clf = make_pipeline(
    PolynomialFeatures(degree=3),
    StandardScaler(),
    LinearSVC(C=10, max_iter=10_000, random_state=42)
)

polynomial_svm_clf.fit(X, y)

Pipeline(steps=[('polynomialfeatures', PolynomialFeatures(degree=3)),
                ('standardscaler', StandardScaler()),
                ('linearsvc',
                 LinearSVC(C=10, max_iter=10000, random_state=42))])

### **5.2.1 다항식 커널**

다항 특성을 추가하는 방법은 간단하고 여러 머신러닝 알고리즘에서 잘 작동

BUT

- 낮은 차수의 다항식 -> 매우 복잡한 데이터셋을 잘 표현하지 못함
- 높은 차수의 다항식 -> 매우 많은 특성을 추가하므로 모델이 느려짐

**커널 트릭** : 실제로 특성을 추가하지 않으면서 매우 높은 차수의 다항 특성을 많이 추가한 것과 같은 결과를 얻는 방법

-> 실제 특성을 추가하지 않으므로 많은 수의 특성 조합이 생기지 않음

In [ ]:
#3차 다항식 커널을 사용한 SVM 분류기
from sklearn.svm import SVC

poly_kernel_svm_clf = make_pipeline(
    StandardScaler(),
    SVC(kernel="poly", degree=3, coef0=1, C=5)
)

poly_kernel_svm_clf.fit(X, y)

Pipeline(steps=[('standardscaler', StandardScaler()),
                ('svc', SVC(C=5, coef0=1, kernel='poly'))])


- 모델이 과대적합 -> degree 감소
- 모델이 과소적합 -> degree 증가
- coef0 : 모델이 높은 차수와 낮은 차수에 얼마나 영향을 받을지 조절

-하이퍼파라미터는 일반적으로 랜덤 서치와 같은 방법으로 자동 튜닝할 수 있음

-하지만 각 하이퍼파라미터가 어떤 기능을 하고 서로 어떻게 상호 작용하는지 이해하면 탐색 범위를 좁힐

### **5.2.2 유사도 특성**

비선형 특성을 다루는 또 다른 방법은 각 샘플이 특정 랜드마크와 얼마나 닮았는지 측정하는 유사도 함수로 새로운 특성을 만드는 것

**가우스 방사 기저 함수**를 유사도 함수로 사용할 수 있다.

가우스 RBF의 값은

- 랜드마크에서 아주 멀리 떨어진 경우 -> 0
- 랜드마크와 같은 위치인 경우 -> 1

사이에서 변하며 종 모양으로 나타난다.

ex) $\gamma=0.3$이고 샘플 $x_1=-1$이 두 랜드마크에서 각각 $1$, $2$만큼 떨어져 있다면 새로운 특성은

$x_2=\exp(-0.3\times1^2)\approx0.74$

$x_3=\exp(-0.3\times2^2)\approx0.30$


랜드마크를 선택하는 간단한 방법은 데이터셋의 모든 샘플 위치에 랜드마크를 설정하는 것

-> 차원이 매우 커져 변환된 훈련 세트가 선형적으로 구분될 가능성이 높아짐

단점

-> 훈련 세트가 매우 크면 매우 많은 특성이 만들어짐

### **5.2.3 가우스 RBF 커널**

유사도 특성은 모든 추가 특성을 계산하려면, 특히 훈련 세트가 큰 경우 연산 비용이 많이 듬.

커널 트릭을 사용하면 유사도 특성을 많이 추가한 것과 비슷한 결과를 얻을 수 있음

In [ ]:
rbf_kernel_svm_clf = make_pipeline(
    StandardScaler(),
    SVC(kernel="rbf", gamma=5, C=0.001)
)

rbf_kernel_svm_clf.fit(X, y)

Pipeline(steps=[('standardscaler', StandardScaler()),
                ('svc', SVC(C=0.001, gamma=5))])

gamma 증가

- 종 모양 그래프가 좁아짐
- 각 샘플의 영향 범위가 작아짐
- 결정 경계가 더 불규칙해지고 각 샘플을 따라 구불구불하게 휘어짐

gamma 감소

- 종 모양 그래프가 넓어짐
- 각 샘플이 넓은 범위에 영향을 줌
- 결정 경계가 더 부드러워짐

=> 하이퍼파라미터 r가 규제 역할을 함
- 과대적합 -> 감마 감소
- 과소적합 -> 감 증가

→ 하이퍼파라미터 C와 비슷

### **5.2.4 계산 복잡도**


LinearSVC는 선형 SVM을 위한 최적화된 알고리즘을 구현한 liblinear 라이브러리를 기반으로 함

- 커널 트릭 지원 X
- 훈련 샘플 수와 특성 수에 거의 선형적으로 증가

훈련 시간 복잡도 -> 대략 $O(m\times n)$

정밀도를 높이면 알고리즘의 수행 시간이 길어진다.

이를 허용 오차 하이퍼파라미터로 조절하며 사이킷런에서는 tol 매개변수를 사용한다.



SVC는 커널 트릭 알고리즘을 구현한 libsvm 라이브러리를 기반

훈련 시간 복잡도는 보통

$O(m^2\times n)$~ $O(m^3\times n)$ 사이

-> 훈련 샘플 수가 수십만 개로 커지면 매우 느려짐  
-> 중소규모의 비선형 훈련 세트에 적합

특성 수에 대해서는 특히 희소 특성인 경우 잘 확장됨



SGDClassifier는 확률적 경사 하강법을 사용

- 점진적 학습 가능
- 메모리를 거의 사용하지 않음
- RAM에 맞지 않는 대규모 데이터셋에서도 훈련 가능

계산 복잡도 - $O(m\times n)$

| 파이썬 클래스 | 시간 복잡도 | 외부 메모리 학습 지원 | 스케일 조정 필요 | 커널 트릭 |
|---|---|---|---|---|
| LinearSVC | $O(m\times n)$ | X | O | X |
| SVC | $O(m^2\times n)$ ~ $O(m^3\times n)$ | X | O | O |
| SGDClassifier | $O(m\times n)$ | O | O | X |

## **5.3 SVM 회귀**

SVM을 분류가 아니라 회귀에 적용할 때는 목표를 바꿈.

분류

-> 제한된 마진 오류 안에서 두 클래스 사이의 도로 폭을 가능한 한 크게 만듦

회귀

-> 제한된 마진 오류 안에서 도로 안에 가능한 한 많은 샘플이 들어가도록 학습



도로의 폭은 하이퍼파라미터 $\epsilon$으로 조절한다.

$\epsilon$을 줄이면 서포트 벡터의 수가 늘어나서 모델이 규제된다.

마진 안에서는 훈련 샘플이 추가되어도 모델의 예측에 영향을 주지 않는다.

-> $\epsilon$에 민감하지 않다

In [ ]:
#LinearSVR을 사용한 선형 SVM 회귀
from sklearn.svm import LinearSVR

# X, y = [...]  # 선형 데이터셋

svm_reg = make_pipeline(
    StandardScaler(),
    LinearSVR(epsilon=0.5, random_state=42)
)

svm_reg.fit(X, y)

Pipeline(steps=[('standardscaler', StandardScaler()),
                ('linearsvr', LinearSVR(epsilon=0.5, random_state=42))])

비선형 회귀 작업을 처리하려면 커널 SVM 모델을 사용

In [ ]:
from sklearn.svm import SVR

# X, y = [...]  # 2차 방정식 데이터셋

svm_poly_reg = make_pipeline(
    StandardScaler(),
    SVR(kernel="poly", degree=2, C=0.01, epsilon=0.1)
)

svm_poly_reg.fit(X, y)

-SVR : SVC의 회귀 버전
-LinearSVR : LinearSVC의 회귀 버전

LinearSVR은 필요한 시간이 훈련 세트의 크기에 비례하여 선형적으로 늘어남

반면 SVR은 SVC처럼 훈련 세트가 커지면 훨씬 느려짐

## **5.4 SVM 이론**

선형 SVM 분류기의 결정 함수

$s=\mathbf{w}^T\mathbf{x}+b$

결정 함수의 값이 0보다 크면 양성 클래스 1을 예측, 그렇지 않으면 음성 클래스 0을 예측한다.

선형 SVM을 훈련하려면 마진 오류 횟수를 제한하면서 도로의 폭을 가능한 한 넓게 만드는 가중치 벡터 $\mathbf{w}$와 편향 $b$를 찾아야 함

도로의 폭을 넓히려면 $\mathbf{w}$를 작게 만들어야 함

-> $\mathbf{w}$가 작을수록 마진은 커짐

편향 $b$는 마진의 크기에는 영향을 미치지 않고 마진의 위치만 이동